# Part 1: Python Basics 🐍
Welcome! Before we dive into Neural Networks, we need to understand the language that powers it: Python. Let's look at the foundational building blocks.

In [ ]:
# 1. Variables and Data Types

# Python dynamically assigns types based on what you store.
my_integer = 10
my_float = 3.14
my_string = "Hello, Deep Learning!"
my_boolean = True

print(f"String: {my_string} | Integer: {my_integer}")

In [ ]:
# 2. Lists and Dictionaries

# Lists store sequences of items.
activations = ["ReLU", "Sigmoid", "Tanh"]
print("First activation function:", activations[0])

# Dictionaries store key-value pairs (like a real dictionary!)
hyperparameters = {
    "learning_rate": 0.01,
    "batch_size": 32
}
print("Learning Rate is:", hyperparameters["learning_rate"])


In [ ]:
# 3. Control Flow (If/Else and Loops)

if hyperparameters["learning_rate"] < 0.1:
    print("Safe learning rate!")
else:
    print("Warning: Learning rate might be too high.")



print("\nIterating through our list:")
for func in activations:
    print(f"- {func}")

In [ ]:
# 4. Functions
# Functions let us reuse code without rewriting it.
def calculate_area(width, height):
    return width * height

print("Area of 5x10 rectangle:", calculate_area(5, 10))

## The Power of Slicing 🍕

Before we move on, we need to talk about slicing. In Machine Learning, we rarely look at just one piece of data at a time. Slicing is how we extract chunks of data from our lists or Tensors.

The syntax is [start : stop : step]. Remember that the stop index is exclusive (it grabs everything up to, but not including, that index).

In [ ]:
# --- Python List Slicing ---
my_list = [0, 10, 20, 30, 40, 50]
print("Original list:", my_list)

# Grab the first three elements (indices 0, 1, 2)
print("First three elements:", my_list[0:3]) 

# Grab everything from index 3 to the end
print("Index 3 onwards:", my_list[3:])

# Grab every second element
print("Every second element:", my_list[::2])

# --- PyTorch Tensor Slicing ---
# Slicing is even more powerful with multi-dimensional Tensors!
import torch

# Let's create a 3x3 matrix
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
print("\nOriginal 3x3 Matrix:\n", matrix)

# Syntax for 2D is [row_slice, column_slice]
# Get the first two rows, and ALL columns (using just a colon ':')
print("\nFirst two rows:\n", matrix[0:2, :])

# Get ALL rows, but only the last column (index 2)
print("\nLast column only:\n", matrix[:, 2])

---
# Part 2: Enter PyTorch 🔥
Now that we know Python, let's introduce **PyTorch**. PyTorch is an open-source machine learning framework. 

Its superpower lies in **Tensors**. Tensors are just like Python lists or NumPy arrays, but they have a special trick: they can run super fast on GPUs (Graphics Processing Units) and can automatically calculate gradients (calculus) for us, which is how neural networks learn!

In [ ]:
# First, we import the PyTorch library
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
from torch.utils.data import TensorDataset, DataLoader
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Let's create our first Tensor!
# Think of it as a grid of numbers.
my_tensor = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("Our Tensor:\n", my_tensor)
print("Tensor Shape:", my_tensor.shape)

---
# Part 3: Building a Multi-Layer Perceptron (MLP) 🧠
An MLP is a classic "feedforward" neural network. It consists of:
1. An **Input Layer** (receiving data)
2. One or more **Hidden Layers** (where the network finds patterns)
3. An **Output Layer** (the final prediction)

Let's build one to classify some dummy data!

In [ ]:
# We define a Neural Network by creating a Python Class
# that inherits from PyTorch's nn.Module.

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        
        # Layer 1: Input to Hidden
        self.fc1 = nn.Linear(input_size, hidden_size)
        # Activation Function: Adds non-linearity (helps it learn complex patterns)
        self.relu = nn.ReLU()
        # Layer 2: Hidden to Output
        self.fc2 = nn.Linear(hidden_size, output_size)
        # Activation for output (Sigmoid squashes the output between 0 and 1 for probabilities)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

# Let's instantiate our model

# E.g., 10 input features, 16 hidden neurons, 1 output prediction (0 or 1)
model = MLP(input_size=30, hidden_size=16, output_size=1)
print(model)

In [ ]:
# Compact Version with raw logits in output
# !!! In PyTorch, output activation functions are often handled inside the loss function instead of the model

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))  # we apply activation here
        x = self.fc2(x)     # no activation in the output layer
        return x

model = MLP(input_size=30, hidden_size=16, output_size=1) # 30 input features as in our X
print(model)

In [ ]:
# Let's add a third hidden layer and see how it performs


---
# Part 4: Training the Network 🏋️‍♀️
To teach our network, we need:
1. **Data:** Inputs and target answers.
2. **A Loss Function:** To measure how wrong the network's guesses are.
3. **An Optimizer:** To update the network's weights and make it smarter based on the loss.
4. **A Loop:** To iteratively process our data 

In [ ]:
# 1. Prepare Our Data

data = load_breast_cancer()

X = data.data
y = data.target

In [ ]:
len(y[y==0])/len(y[y==1])

In [ ]:

# First split off test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Then split train into train/val
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create a Dataset and a DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=16, shuffle=False)

In [ ]:
# Shape of our dataset
data['data'].shape

In [ ]:
# 2. Define the Loss Function and Optimizer

#criterion = nn.BCELoss() # Binary Cross Entropy Loss This requires the model output to already be probabilities in range [0,1]
criterion = nn.BCEWithLogitsLoss() # Returns raw Logits instead of probabilities


# Optimizer (updates model parameters)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50

print("Starting Training...")

for epoch in range(epochs):

    # ---------------- TRAINING PHASE ----------------
    # Set model to training mode (enables dropout or batchnorm behavior if we included it)
    model.train()

    train_loss = 0

    # Iterate over minibatches of training data
    for X_batch, y_batch in train_loader:

        # Reset gradients from previous step
        optimizer.zero_grad()

        # Forward pass: compute predictions (logits)
        predictions = model(X_batch)

        # Compute training loss
        loss = criterion(predictions, y_batch)

        # Backpropagation: compute gradients
        loss.backward()

        # Update model weights
        optimizer.step()

        # Accumulate loss for reporting
        train_loss += loss.item()


    # ---------------- VALIDATION PHASE ----------------
    # Set model to evaluation mode (disables dropout, batchnorm updates)
    model.eval()

    val_loss = 0
    all_preds = []
    all_labels = []

    # Disable gradient computation (saves memory and computation)
    with torch.no_grad():

        # Iterate over validation batches
        for X_batch, y_batch in val_loader:

            # Forward pass only (no weight updates)
            predictions = model(X_batch)

            # Compute validation loss
            loss = criterion(predictions, y_batch)

            val_loss += loss.item()
            
            # Convert logits → class indices
            predicted_classes = (torch.sigmoid(predictions) > 0.5).long().squeeze() # long converts boolean → integer tensor, squeeze removes unnec dims

            all_preds.extend(predicted_classes.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())



    # ---------------- LOGGING ----------------
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss/len(train_loader):.4f} | "
            f"Val Loss: {val_loss/len(val_loader):.4f}"
        )

In [ ]:
# after validation loop, replace f1_score with:
print(classification_report(all_labels, all_preds, zero_division=0))

### Let's see the results on the unseen data (Test Set)

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_classes = (torch.sigmoid(predictions) > 0.5).long().squeeze() 
        all_preds.extend(predicted_classes.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

print(classification_report(all_labels, all_preds, zero_division=0))

# Exercise: Build Your First Activation Function 🧠 

**The Scenario:**
In Deep Learning, a common "activation function" is called **ReLU** (Rectified Linear Unit). It has a very simple job: it looks at a number, and if the number is negative, it changes it to `0`. If the number is positive, it leaves it alone.

**Your Task:**
Write a Python function called `manual_relu` that processes a list of numbers according to these steps:
1. Create an empty list to store the results.
2. Loop through the `inputs` list.
3. Check if the number is less than `0`.
   * If it is, append `0` to your results list.
   * If it is not, append the original number.
4. Return the results list.

In [ ]:
def manual_relu(inputs):
    # Write your code here!
    pass

### 🧪 Test Your Code
**Do not change the code below.** Run the cell to see if you succeeded!

In [ ]:
test_data = [-5, 2, -1, 8, 0, -3.5, 10]
expected_output = [0, 2, 0, 8, 0, 0, 10]

student_output = manual_relu(test_data)

print(f"Your Output:     {student_output}")
print(f"Expected Output: {expected_output}\n")

if student_output == expected_output:
    print("✅ Success! You just built a neural network component!")
else:
    print("❌ Not quite yet. Check your loop and conditions!")

# Exercise 2: Measuring the Mistake (Manual Loss) 📉 

**The Scenario**:
In Part 4, we used a Loss Function (the criterion) to tell the model how far off its guesses were. One of the most common ways to measure error is the **Squared Error**. You take the difference between the truth and the guess, and square it to ensure the error is always a positive number.

---

**Your Task**:

Write a function called `manual_squared_error` that compares a list of "true" labels to a list of "predicted" scores:

1. Create a variable `total_error` and set it to `0`.

2. Use a `for` loop to iterate through the indices of the lists  
  *(Hint: use `range(len(true_values))`)*.

3. For each pair of values:
  - Subtract the predicted value from the true value to find the difference.
  - Square that difference *(in Python, use `** 2`)*.
  - Add that squared value to your `total_error`.

4. Return the `total_error`.

In [ ]:
def manual_squared_error(true_values, predicted_values):
    # Your code here!
    # Hint: Use 'range(len(true_values))' to loop through indices
    pass

### 🧪 Test Your Code
**Do not change the code below.** Run the cell to see if you succeeded!

In [ ]:
# Test data: True values vs what the model guessed
truths = [1, 0, 1, 1]
guesses = [0.8, 0.3, 0.9, 0.2]

# Calculation: (1-0.8)^2 + (0-0.3)^2 + (1-0.9)^2 + (1-0.2)^2
# = 0.04 + 0.09 + 0.01 + 0.64 = 0.78

student_loss = manual_squared_error(truths, guesses)

if student_loss is not None and round(student_loss, 2) == 0.78:
    print(f"Your Loss: {student_loss:.2f}")
    print("✅ Excellent! You just calculated how a brain learns from its mistakes.")
else:
    print(f"Your Loss: {student_loss}")
    print("❌ Not quite. Remember to square the difference for each pair!")